In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Libraries loaded.')

In [ ]:
# ── Generate realistic synthetic loan dataset ────────────────────────────────
n = 3000

age              = np.random.randint(21, 65, n)
income           = np.random.normal(55000, 18000, n).clip(15000, 200000).astype(int)
loan_amount      = np.random.normal(12000, 6000, n).clip(1000, 50000).astype(int)
loan_term        = np.random.choice([12, 24, 36, 48, 60], n)
interest_rate    = np.round(np.random.uniform(5.0, 22.0, n), 2)
credit_score     = np.random.randint(300, 850, n)
employment_years = np.random.randint(0, 30, n)
num_prev_loans   = np.random.randint(0, 8, n)
missed_payments  = np.random.randint(0, 5, n)
debt_to_income   = np.round(loan_amount / income, 4)
loan_purpose     = np.random.choice(
    ['business', 'personal', 'education', 'home_improvement', 'medical'], n
)

# Default probability driven by meaningful features
default_prob = (
    0.35 * (credit_score < 580).astype(int) +
    0.20 * (missed_payments >= 2).astype(int) +
    0.15 * (debt_to_income > 0.4).astype(int) +
    0.10 * (interest_rate > 18).astype(int) +
    0.10 * (employment_years < 2).astype(int) +
    0.05 * (income < 25000).astype(int) +
    np.random.uniform(0, 0.15, n)
)
default = (default_prob > 0.45).astype(int)

df = pd.DataFrame({
    'age': age,
    'income': income,
    'loan_amount': loan_amount,
    'loan_term': loan_term,
    'interest_rate': interest_rate,
    'credit_score': credit_score,
    'employment_years': employment_years,
    'num_prev_loans': num_prev_loans,
    'missed_payments': missed_payments,
    'debt_to_income': debt_to_income,
    'loan_purpose': loan_purpose,
    'default': default
})

df.to_csv('../data/loan_data.csv', index=False)
print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df.default.mean():.2%}')
df.head()

In [ ]:
# ── Exploratory Data Analysis ────────────────────────────────────────────────
print('=== Dataset Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Summary Stats ===')
df.describe().round(2)

In [ ]:
# Default rate by loan purpose
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

purpose_default = df.groupby('loan_purpose')['default'].mean().sort_values(ascending=False)
purpose_default.plot(kind='bar', ax=axes[0], color='#1A56DB', edgecolor='white')
axes[0].set_title('Default Rate by Loan Purpose', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Default Rate')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Credit score distribution by default
df[df.default==0]['credit_score'].hist(ax=axes[1], alpha=0.6, bins=30, color='#1A56DB', label='No Default')
df[df.default==1]['credit_score'].hist(ax=axes[1], alpha=0.6, bins=30, color='#E53E3E', label='Default')
axes[1].set_title('Credit Score Distribution by Default Status', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Credit Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eda_plot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric features only)
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=np.number)
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))
sns.heatmap(
    numeric_df.corr(), mask=mask, annot=True, fmt='.2f',
    cmap='Blues', linewidths=0.5, annot_kws={'size': 9}
)
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────────
df_model = df.copy()

# Encode categorical
le = LabelEncoder()
df_model['loan_purpose_enc'] = le.fit_transform(df_model['loan_purpose'])

# Save encoder classes for API use
purpose_classes = list(le.classes_)
print('Loan purpose classes (order matters for encoding):', purpose_classes)

# Engineered features
df_model['monthly_payment_burden'] = (
    (df_model['loan_amount'] * df_model['interest_rate'] / 100) / df_model['loan_term']
) / (df_model['income'] / 12)

df_model['credit_risk_flag'] = (df_model['credit_score'] < 580).astype(int)
df_model['high_missed_flag']  = (df_model['missed_payments'] >= 2).astype(int)

FEATURES = [
    'age', 'income', 'loan_amount', 'loan_term', 'interest_rate',
    'credit_score', 'employment_years', 'num_prev_loans',
    'missed_payments', 'debt_to_income', 'loan_purpose_enc',
    'monthly_payment_burden', 'credit_risk_flag', 'high_missed_flag'
]

X = df_model[FEATURES]
y = df_model['default']

print(f'Feature matrix: {X.shape}')
X.head()

In [ ]:
# ── Train / Test Split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train default rate: {y_train.mean():.2%} | Test default rate: {y_test.mean():.2%}')

In [ ]:
# ── Model Training ───────────────────────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=150,
        max_depth=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

pipeline.fit(X_train, y_train)

# Cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc')
print(f'5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────
y_pred      = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f}')

In [ ]:
# Confusion matrix + ROC curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No Default', 'Default'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='#1A56DB', lw=2, label=f'ROC Curve (AUC = {auc:.3f})')
axes[1].plot([0,1],[0,1],'--', color='grey', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/evaluation_plot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importances = pipeline.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
feat_imp.plot(kind='barh', color='#1A56DB', edgecolor='white')
plt.title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save model + metadata ────────────────────────────────────────────────────
import json

joblib.dump(pipeline, '../app/model/loan_model.pkl')

metadata = {
    'features': FEATURES,
    'loan_purpose_classes': purpose_classes,
    'model_type': 'RandomForestClassifier',
    'roc_auc': round(float(roc_auc_score(y_test, y_pred_prob)), 4),
    'test_accuracy': round(float((y_pred == y_test).mean()), 4)
}

with open('../app/model/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Model saved to ../app/model/loan_model.pkl')
print('Metadata saved to ../app/model/metadata.json')
print(json.dumps(metadata, indent=2))